# 氷結晶粒径予測パイプライン（データセット作成 / Colab用）
crop画像とラベルCSVを作成するところまでを Colab で行う。
学習・評価は VSCode 側（`vscode_project/`）で行う。

| セル | 内容 |
|------|------|
| Step 0 | セットアップ（ライブラリ・定数・Driveマウント） |
| Step 1 | データセット作成（クロップ画像 + CSV） |
| Step 2 | エクスポート（zip化してダウンロード → VSCode側の `data/` に展開） |

> **注意**: ランタイム → GPU は不要（crop処理はCPUで十分）。GPUを使う場合はランタイムで選択してください。

## Step 0: セットアップ

In [ ]:
import os, re, shutil
import pandas as pd
from PIL import Image
from collections import defaultdict

from google.colab import drive
drive.mount('/content/drive')

# ===== パス設定（ここを実際のパスに変更） =====
IMAGE_ROOT  = "/content/drive/MyDrive/研究/画像データ"
CSV_ROOT    = "/content/drive/MyDrive/研究/Excelデータ"
OUTPUT_DIR  = "/content/crops"
DATASET_CSV = "/content/dataset.csv"

# ===== スケール設定 =====
MAG_TO_UM_PER_PIXEL = {40: 0.088725, 20: 0.17353, 10: 0.34392}
BASE_UM_PER_PIXEL   = MAG_TO_UM_PER_PIXEL[40]

COL_X1          = "矩形領域(左上:x)"
COL_Y1          = "矩形領域(左上:y)"
COL_X2          = "矩形領域(右下:x)"
COL_Y2          = "矩形領域(右下:y)"
COL_DIAMETER    = "円相当径"
IMG_EXTS        = (".bmp", ".tif", ".tiff", ".jpg", ".png")
EXCLUDE_MINUTES = {"0"}

print("セットアップ完了")

## Step 1: データセット作成
`OUTPUT_DIR` にクロップ画像を、`DATASET_CSV` にラベル CSV を出力する。
**既に実行済みの場合はスキップ可。**

In [ ]:
def normalize_folder_name(name):
    return re.sub(r"[（）()・\s]", "", name)

def get_minute_mag_key(filename):
    m = re.search(r"(\d+)分.*?(\d+)倍", filename)
    return m.groups() if m else None

def find_diameter_col(columns):
    for c in columns:
        if COL_DIAMETER in str(c):
            return c
    raise ValueError(f"円相当径列が見つかりません: {list(columns)}")

def make_dataset():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    img_dirs = {normalize_folder_name(d): d
                for d in os.listdir(IMAGE_ROOT)
                if os.path.isdir(os.path.join(IMAGE_ROOT, d))}
    csv_dirs = {normalize_folder_name(d): d
                for d in os.listdir(CSV_ROOT)
                if os.path.isdir(os.path.join(CSV_ROOT, d))}

    common = set(img_dirs) & set(csv_dirs)
    print(f"画像フォルダ: {len(img_dirs)}件  CSVフォルダ: {len(csv_dirs)}件  対応: {len(common)}件")

    rows, crop_id = [], 0

    for key in sorted(common):
        image_dir = os.path.join(IMAGE_ROOT, img_dirs[key])
        csv_dir   = os.path.join(CSV_ROOT,   csv_dirs[key])
        print(f"\n=== {img_dirs[key]} ===")

        images = defaultdict(list)
        for f in os.listdir(image_dir):
            if f.lower().endswith(IMG_EXTS):
                k = get_minute_mag_key(f)
                if k:
                    images[k].append(f)

        csvs = defaultdict(list)
        for f in os.listdir(csv_dir):
            if f.lower().endswith(".csv"):
                k = get_minute_mag_key(f)
                if k:
                    csvs[k].append(f)

        for k in sorted(set(images) & set(csvs)):
            minute, _ = k
            if minute in EXCLUDE_MINUTES:
                print(f"  [除外] {minute}分")
                continue

            for img_f, csv_f in zip(sorted(images[k]), sorted(csvs[k])):
                image_path = os.path.join(image_dir, img_f)
                csv_path   = os.path.join(csv_dir,   csv_f)

                m = re.search(r"(\d+)倍", img_f)
                mag = int(m.group(1)) if m else None
                if mag not in MAG_TO_UM_PER_PIXEL:
                    continue
                um_per_pixel = MAG_TO_UM_PER_PIXEL[mag]

                try:
                    df  = pd.read_csv(csv_path, encoding="cp932")
                    dcol = find_diameter_col(df.columns)
                    img  = Image.open(image_path)
                except Exception as e:
                    print(f"  [スキップ] {e}")
                    continue

                n_crops = 0
                for _, crystal in df.iterrows():
                    try:
                        x1 = int(crystal[COL_X1]); y1 = int(crystal[COL_Y1])
                        x2 = int(crystal[COL_X2]); y2 = int(crystal[COL_Y2])
                        d  = float(crystal[dcol])
                    except (KeyError, ValueError, TypeError):
                        continue

                    left, right = sorted([x1, x2])
                    top, bottom = sorted([y1, y2])
                    if right <= left or bottom <= top:
                        continue

                    crop  = img.crop((left, top, right, bottom))
                    scale = um_per_pixel / BASE_UM_PER_PIXEL
                    crop  = crop.resize((max(1, round(crop.width  * scale)),
                                         max(1, round(crop.height * scale))))

                    crop_path = os.path.join(OUTPUT_DIR, f"crop_{crop_id:06d}.png")
                    crop.save(crop_path)

                    rows.append({"filepath": os.path.basename(crop_path),
                                 "label_um": d * um_per_pixel,
                                 "source_folder": img_dirs[key],
                                 "source_image":  img_f})
                    crop_id += 1
                    n_crops += 1

                print(f"  {img_f} → {n_crops}件")

    out_df = pd.DataFrame(rows)
    out_df.to_csv(DATASET_CSV, index=False, encoding="utf-8-sig")
    print(f"\n完了: {len(out_df)}件 → {DATASET_CSV}")

make_dataset()

## Step 2: エクスポート（VSCode側へ持ち出す）
`crops/` と `dataset.csv` を1つのzipにまとめてダウンロードする。
VSCode側の `vscode_project/data/` にそのまま展開すれば、`data/crops/`・`data/dataset.csv` の配置になる。

In [ ]:
EXPORT_DIR = "/content/export"
os.makedirs(EXPORT_DIR, exist_ok=True)

shutil.copytree(OUTPUT_DIR, os.path.join(EXPORT_DIR, "crops"), dirs_exist_ok=True)
shutil.copy(DATASET_CSV, os.path.join(EXPORT_DIR, "dataset.csv"))

zip_path = shutil.make_archive("/content/crystal_dataset", "zip", EXPORT_DIR)
print(f"作成: {zip_path}")

from google.colab import files
files.download(zip_path)